# AlphaGo — Mastering the Game of Go with Deep Neural Networks and Tree Search (2016)

_Papers · Reinforcement Learning_

**Two neural networks (one picks moves, one scores positions) guide a tree search that beat a human Go professional.**

---

This notebook is a practice scaffold for the **AlphaGo — Mastering the Game of Go with Deep Neural Networks and Tree Search (2016)** lesson. We build it up one step at a time: walk through a single PUCT selection step, then run a tiny Monte-Carlo tree search and watch the visit counts pile onto the best move. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

We build it in three steps: (1) score one node's children with the PUCT rule $Q + u$ to see how AlphaGo balances proven value against the policy prior, (2) set up a toy one-move game with hidden move qualities, and (3) run a tiny MCTS loop and watch the visit counts concentrate on the best move.

### Step 1 — Score one PUCT selection step (Eq 5)

At each node, AlphaGo picks the move maximizing $Q + u$, where $Q$ is the action value learned so far and $u = c\,P/(1+N)$ is an exploration bonus from the policy prior $P$ that decays as a move is visited more (larger $N$). Here three children compete: A has a high proven $Q$, B has a high prior but few visits. We compute each score and see which Eq (5) selects.

In [ ]:
import numpy as np

# WORKED EXAMPLE: one PUCT selection step (Eq 5). a = argmax(Q + u),
# u = c * P / (1 + N).  Three child moves of one node.  c = 1.5.
c = 1.5
moves = ["A", "B", "C"]
Q = np.array([0.60, 0.10, 0.20])      # action values so far
P = np.array([0.20, 0.50, 0.30])      # policy-network priors (sum to 1)
N = np.array([8,    1,    3   ])      # visit counts

u = c * P / (1 + N)                   # exploration bonus (Eq 5)
score = Q + u
for m, q, p, n, ui, si in zip(moves, Q, P, N, u, score):
    print("%s: Q=%.2f P=%.2f N=%d  u=%.4f  Q+u=%.4f" % (m, q, p, n, ui, si))
print("argmax (Eq 5 selects) ->", moves[int(np.argmax(score))])
# A: Q=0.60 P=0.20 N=8  u=0.0333  Q+u=0.6333
# B: Q=0.10 P=0.50 N=1  u=0.3750  Q+u=0.4750
# C: Q=0.20 P=0.30 N=3  u=0.1125  Q+u=0.3125
# argmax (Eq 5 selects) -> A      (A's proven value beats B's high prior)

### Step 2 — Set up a toy one-move game and warm-start the children

We model a single decision with three moves of differing *hidden* true win-probability (A is best, B close behind). The policy prior $P$ is what the policy network would suggest. We first expand each child a few times (a warm start) so every move has a defined $Q$ before the guided search begins. This is our made-up illustration, not Go and not an AlphaGo result.

In [ ]:
rng = np.random.default_rng(0)
prior   = np.array([0.40, 0.35, 0.25])      # policy prior P
truewin = np.array([0.62, 0.52, 0.45])      # hidden true win-prob (A best, B close)
c2 = 1.2
N2 = np.zeros(3)                            # visit counts per move
W2 = np.zeros(3)                            # total wins per move

for a in range(3):                          # warm start: expand each child 5x
    for _ in range(5):
        W2[a] += rng.binomial(1, truewin[a])
        N2[a] += 1

print("after warm start  visits =", N2.astype(int).tolist(), " wins =", W2.astype(int).tolist())

### Step 3 — Run the guided MCTS loop

Now we repeat the AlphaGo cycle: SELECT the move maximizing $\hat{Q} + u$ (Eq 5), EVALUATE the leaf with a noisy rollout (a Bernoulli draw from the hidden win-prob), and BACK UP the visit count and win total (Eq 7, 8). AlphaGo finally plays the *most-visited* root move. Watch the visits pile onto move A, the best one.

In [ ]:
for i in range(1, 121):                     # 120 guided simulations
    Qhat = W2 / N2                          # mean leaf value so far (Eq 8)
    u2 = c2 * prior / (1 + N2)              # bonus (Eq 5)
    a = int(np.argmax(Qhat + u2))           # SELECT
    W2[a] += rng.binomial(1, truewin[a])    # EVALUATE leaf (noisy rollout)
    N2[a] += 1                              # BACK UP visit count (Eq 7)
    if i in (10, 40, 80, 120):
        print("after %3d sims  visits =" % i, N2.astype(int).tolist())

print("final Q =", [round(W2[k] / N2[k], 3) for k in range(3)])
print("chosen (most visited) move:", moves[int(np.argmax(N2))])
# after  10 sims  visits = [10, 5, 10]
# after  40 sims  visits = [38, 5, 12]
# after  80 sims  visits = [78, 5, 12]
# after 120 sims  visits = [118, 5, 12]
# final Q = [0.669, 0.4, 0.5]
# chosen (most visited) move: A
# Visit counts pile onto move A (the best), as AlphaGo plays the most-visited
# root move. Toy numbers, OUR illustration -- NOT Go, NOT an AlphaGo result.

## Visualize the data & results

_AlphaGo's MCTS selects moves by argmax(Q + u), u ~ P/(1+N), and finally plays the MOST-VISITED move. If we run a tiny MCTS on a toy one-move game where move A is best, do the visit counts concentrate on A as simulations accumulate?_

### Step 1 — Set up the toy game again for a standalone run

This panel re-imports NumPy and rebuilds the same toy one-move game — the policy prior, the hidden true win-probabilities, and the warm-start expansion — so it runs on its own. Identical setup to the reference implementation above.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# Toy one-move game: three moves, hidden true win-probs (A best). OUR illustration.
prior   = np.array([0.40, 0.35, 0.25])    # policy prior P (the "policy network")
truewin = np.array([0.62, 0.52, 0.45])    # hidden true win-prob revealed by rollouts
c = 1.2
N = np.zeros(3)                           # visits
W = np.zeros(3)                           # total wins

for a in range(3):                        # warm start: expand each child 5x
    for _ in range(5):
        W[a] += rng.binomial(1, truewin[a])
        N[a] += 1

### Step 2 — Run the simulations and snapshot the visit counts

We run 120 guided MCTS simulations, recording the visit counts at four checkpoints. Each simulation selects by $\hat{Q}+u$, evaluates with a noisy rollout, and backs up — exactly the cycle from the reference implementation.

In [ ]:
snaps = {}
for i in range(1, 121):                   # 120 guided MCTS simulations
    Qhat = W / N                          # mean leaf value (Eq 8)
    u = c * prior / (1 + N)               # exploration bonus (Eq 5)
    a = int(np.argmax(Qhat + u))          # SELECT argmax(Q + u)
    W[a] += rng.binomial(1, truewin[a])   # EVALUATE leaf (noisy rollout)
    N[a] += 1                             # BACK UP (Eq 7)
    if i in (10, 40, 80, 120):
        snaps[i] = N.astype(int).tolist()

### Step 3 — Read off where the search concentrated

We print the snapshots, the final $Q$ estimates, and the most-visited move. The visit counts concentrate on move A — the best one — confirming that AlphaGo's most-visited-move rule lands on the strongest option as simulations accumulate. Our toy run, not Go and not AlphaGo's numbers.

In [ ]:
for i in (10, 40, 80, 120):
    print("after %3d sims  visits =" % i, snaps[i])
print("final Q =", [round(W[k] / N[k], 3) for k in range(3)])
print("total visits =", int(N.sum()), " most-visited move index:", int(np.argmax(N)))
# after  10 sims  visits = [10, 5, 10]
# after  40 sims  visits = [38, 5, 12]
# after  80 sims  visits = [78, 5, 12]
# after 120 sims  visits = [118, 5, 12]
# final Q = [0.669, 0.4, 0.5]
# total visits = 135  most-visited move index: 0   (move A, the best)
# Visit counts concentrate on the best move. OUR toy run, NOT Go, NOT AlphaGo's numbers.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** One PUCT selection step. A node has two moves. Move X: $Q = 0.55$, $P = 0.30$, $N = 9$.
            Move Y: $Q = 0.40$, $P = 0.60$, $N = 1$. Use $u = c\,P/(1+N)$ with $c = 1.0$. (a) Which move does
            Eq (5) select? (b) After the search visits the chosen move once more, recompute its bonus and say
            whether the selection could flip.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute $u_X = 1.0 \times 0.30 / (1+9) = 0.030$, so $Q+u = 0.55 + 0.030 = 0.580$. — _Eq (5) scores each move by its action value plus the decaying prior bonus._
- Compute $u_Y = 1.0 \times 0.60 / (1+1) = 0.300$, so $Q+u = 0.40 + 0.300 = 0.700$. — _Y has a high prior $P$ and only one visit, so its bonus is large._
- Select the larger: $0.700 > 0.580$, so the search picks Y. After visiting Y once more, $N_Y = 2$, so $u_Y = 0.60/3 = 0.200$ and its score drops to $0.600$ &mdash; still above X's $0.580$, but closer. — _Each visit shrinks the bonus, so a high-prior move loses its edge as it is explored; eventually $Q$ decides._

**Answer:** (a) Move Y wins this step: $Q+u = 0.700$ versus X's $0.580$, because Y's large prior and
                 single visit give it a big exploration bonus. (b) After one more visit to Y, $N_Y = 2$ and its
                 score falls to $0.40 + 0.60/3 = 0.600$ &mdash; still ahead of X ($0.580$), but the gap has
                 nearly closed. A few more visits to Y (or a low value found below it) would let X overtake. This
                 is the bonus decaying as $1/(1+N)$: exploration first, exploitation later.

</details>

**Problem 2.** Blending the leaf evaluation. A leaf has value-network estimate $v_\theta(s_L) = 0.8$ (a win
            looks likely) and a rollout result $z_L = -1$ (the quick played-out game was lost). Using Eq (6),
            compute the leaf evaluation $V(s_L)$ for $\lambda = 0$, $\lambda = 0.5$, and $\lambda = 1$. Which
            does AlphaGo use, and what does the disagreement tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\lambda = 0$: $V = (1-0)(0.8) + 0(-1) = 0.8$ &mdash; trust the value network only. — _At $\lambda = 0$ the rollout is ignored; Eq (6) reduces to $v_\theta$._
- $\lambda = 1$: $V = (1-1)(0.8) + 1(-1) = -1$ &mdash; trust the rollout only. — _At $\lambda = 1$ the value network is ignored; Eq (6) reduces to $z_L$._
- $\lambda = 0.5$: $V = 0.5(0.8) + 0.5(-1) = 0.4 - 0.5 = -0.1$ &mdash; the blend AlphaGo uses. — _The mixed evaluation ($\lambda = 0.5$) performed best (&sect;5); it averages two error-prone signals._

**Answer:** $V = 0.8$ ($\lambda=0$), $V = -0.1$ ($\lambda=0.5$), $V = -1$ ($\lambda=1$). AlphaGo uses
                 the blend $\lambda = 0.5$, giving $-0.1$ &mdash; a cautious near-even score. The disagreement
                 (the value net likes the position, the rollout lost) is exactly why the blend helps: the two
                 estimators err in different ways, and averaging them is more reliable than either alone. The
                 paper reports $\lambda = 0.5$ winning $\geq 95\%$ against the value-only and rollout-only
                 variants (&sect;5).

</details>

**Problem 3.** Ablation &mdash; remove the policy prior. Suppose you set every prior $P(s,a)$ equal (a uniform
            policy), so the policy network no longer guides the search, while keeping everything else (value net,
            rollouts, back-up). Predict what happens to the search's efficiency and to AlphaGo's strength, and
            tie it to a number in the paper.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- In Eq (5), the bonus $u \propto P/(1+N)$ now has the same $P$ for every move, so early
                 selection is driven only by visit counts &mdash; the search no longer focuses on the moves a
                 strong player would consider. — _The policy prior is what narrows the breadth of the search to promising moves; removing it makes the search spread effort over all legal moves._
- With ~250 legal moves in Go and a fixed simulation budget, each move gets far fewer simulations,
                 so $Q$ estimates are noisier and the most-visited move is less reliable. — _Breadth reduction is half of AlphaGo's design (&sect;intro); without it the tree is too wide to evaluate well in the time given._
- The paper's own evidence: the policy-network strength tracks playing strength tightly &mdash;
                 "small improvements in accuracy led to large improvements in playing strength" (&sect;1,
                 Figure 2a). A uniform (useless) policy is the extreme of low accuracy, so strength should fall
                 sharply. — _Figure 2a shows AlphaGo's win rate rising with the policy network's prediction accuracy, so degrading the policy to uniform predicts a large strength drop._

**Answer:** The search loses its move-selection guidance: every move starts with the same bonus, so
                 simulations spread thinly across all ~250 legal moves instead of concentrating on the handful a
                 strong player would weigh. Visit counts and $Q$ estimates become noisy under a fixed budget, and
                 the most-visited move is less trustworthy &mdash; so AlphaGo gets substantially weaker. This
                 matches Figure 2a, where "small improvements in [policy] accuracy led to large improvements in
                 playing strength" (&sect;1): driving the policy to uniform is the worst case of low accuracy. The
                 policy network's job is breadth reduction; without it, the search cannot cover the tree well.

</details>